In [7]:
import pickle

# path = '../../crypto-data/filter_hour_cache/usdt_data_2025-04.pkl'
# path = '/home/craz/crypto/crypto-data/available_symbol/2025-03_symbols.pkl'
# path = '/home/craz/crypto/crypto-data/pickle_cache/ZKJUSDT_full_data.pkl'
path = '/home/craz/crypto/crypto-trading/strategy/backtest/factor_data/cnn_10_03_72h_v3/factor_2025-04.pkl'
# path = '/home/craz/crypto/crypto-trading/strategy/backtest/factor_data/cnn_10_03_72h_v1/factor_2025-04.pkl'
with open(path, "rb") as f:        # 二进制读取
    df = pickle.load(f)




In [5]:
# (0.510529*0.64+0.170022*0.8+0.143674)/(0.64+0.8+1)
# df.head(100)

In [9]:
import pandas as pd
# 假设 df 的 index 是 MultiIndex(index=['open_time','symbol'])
sym = "VOXELUSDT"
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
sub = df.xs(sym, level="symbol")                 # 得到该 symbol 的时间序列（index=open_time）

sub

,factor_value
open_time,
2025-04-01 00:00:00,0.034858
2025-04-01 01:00:00,0.017218
2025-04-01 02:00:00,-0.047002
2025-04-01 03:00:00,0.004589
2025-04-01 04:00:00,-0.048650
2025-04-01 05:00:00,-0.031293
2025-04-01 06:00:00,0.112744
2025-04-01 07:00:00,-0.000359
2025-04-01 08:00:00,0.038668


In [22]:
import pickle
import pandas as pd

# Read a sample factor file to understand the data structure
with open('/home/craz/crypto/crypto-trading/strategy/backtest/factor_data/cnn_04_09_16h_v2/factor_2024-01.pkl', 'rb') as f:
    factor_data = pickle.load(f)

print("Type of factor_data:", type(factor_data))
print("Shape:", factor_data.shape if hasattr(factor_data, 'shape') else 'No shape attribute')

if isinstance(factor_data, pd.DataFrame):
    print("Columns:", factor_data.columns.tolist())
    print("Index type:", type(factor_data.index))
    print("First few rows:")
    print(factor_data.head())
    print("Sample timestamp format:", factor_data.index[0] if len(factor_data) > 0 else "No data")
elif isinstance(factor_data, dict):
    print("Dictionary keys:", list(factor_data.keys())[:10])
    if len(factor_data) > 0:
        first_key = list(factor_data.keys())[0]
        print(f"Sample value type for key {first_key}: {type(factor_data[first_key])}")
        print(f"Sample value: {factor_data[first_key]}")
else:
    print("Data type not recognized, showing first few elements:")
    print(factor_data[:5] if hasattr(factor_data, '__getitem__') else str(factor_data)[:500])

Type of factor_data: <class 'pandas.core.frame.DataFrame'>
Shape: (174937, 1)
Columns: ['factor_value']
Index type: <class 'pandas.core.indexes.multi.MultiIndex'>
First few rows:
                          factor_value
open_time  symbol                     
2024-01-01 1000BONKUSDT       0.022797
           1000FLOKIUSDT     -0.011883
           1000LUNCUSDT      -0.007918
           1000PEPEUSDT      -0.005267
           1000RATSUSDT      -0.019753
Sample timestamp format: (Timestamp('2024-01-01 00:00:00'), '1000BONKUSDT')


In [23]:
import pandas as pd
import numpy as np
from typing import Union

def calculate_wma_weights(time_span_hours: int, forgetting_factor: float) -> np.ndarray:
    """
    Calculate weights for WMA based on time span and forgetting factor.
    
    Args:
        time_span_hours: Number of hours to look back
        forgetting_factor: Forgetting factor (e.g., 0.8)
    
    Returns:
        np.ndarray: Array of weights, ordered from oldest to newest
    """
    # Generate powers from time_span_hours-1 down to 0
    # For time_span_hours=3, powers = [2, 1, 0] (for t-2h, t-1h, t)
    powers = np.arange(time_span_hours - 1, -1, -1)
    
    # Calculate weights: [forgetting_factor^2, forgetting_factor^1, forgetting_factor^0]
    weights = forgetting_factor ** powers
    
    # Normalize weights to sum to 1
    normalized_weights = weights / weights.sum()
    
    return normalized_weights

# Test the weight calculation
time_span = 3  # 2 hours look-back + current hour = 3 periods
forgetting_factor = 0.8

weights = calculate_wma_weights(time_span, forgetting_factor)
print(f"Time span: {time_span} hours (look back {time_span-1} hours)")
print(f"Forgetting factor: {forgetting_factor}")
print(f"Raw weights: {forgetting_factor**2:.3f}, {forgetting_factor**1:.3f}, {forgetting_factor**0:.3f}")
print(f"Normalized weights: {weights}")
print(f"Sum of weights: {weights.sum():.6f}")

# Verify the formula matches user's example:
# For 12 PM with 2h lookback: (10AM*0.8^2 + 11AM*0.8 + 12PM) / (0.8^2 + 0.8 + 1)
raw_weights = np.array([0.8**2, 0.8**1, 0.8**0])
denominator = raw_weights.sum()
manual_normalized = raw_weights / denominator
print(f"\nManual calculation verification:")
print(f"Raw weights: {raw_weights}")
print(f"Denominator: {denominator}")
print(f"Manual normalized: {manual_normalized}")
print(f"Match? {np.allclose(weights, manual_normalized)}")

Time span: 3 hours (look back 2 hours)
Forgetting factor: 0.8
Raw weights: 0.640, 0.800, 1.000
Normalized weights: [0.26229508 0.32786885 0.40983607]
Sum of weights: 1.000000

Manual calculation verification:
Raw weights: [0.64 0.8  1.  ]
Denominator: 2.4400000000000004
Manual normalized: [0.26229508 0.32786885 0.40983607]
Match? True


In [24]:
def calculate_wma_for_symbol_timestamp(
    factor_data: pd.DataFrame,
    symbol: str,
    current_timestamp: pd.Timestamp,
    time_span_hours: int,
    forgetting_factor: float
) -> float:
    """
    Calculate WMA for a specific symbol at a specific timestamp.
    
    Args:
        factor_data: DataFrame with MultiIndex (open_time, symbol) and 'factor_value' column
        symbol: Symbol to calculate WMA for
        current_timestamp: Current timestamp
        time_span_hours: Number of hours to look back (including current)
        forgetting_factor: Forgetting factor for exponential decay
    
    Returns:
        float: WMA value, or NaN if insufficient data
    """
    # Get data for this symbol
    try:
        symbol_data = factor_data.xs(symbol, level='symbol')
    except KeyError:
        return np.nan
    
    # Generate timestamps to look for (from oldest to newest)
    timestamps_to_find = []
    for i in range(time_span_hours - 1, -1, -1):  # [time_span-1, time_span-2, ..., 1, 0]
        ts = current_timestamp - pd.Timedelta(hours=i)
        timestamps_to_find.append(ts)
    
    # Extract factor values for these timestamps
    factor_values = []
    found_timestamps = []
    
    for ts in timestamps_to_find:
        if ts in symbol_data.index:
            factor_values.append(symbol_data.loc[ts, 'factor_value'])
            found_timestamps.append(ts)
        else:
            # If any required timestamp is missing, return NaN
            return np.nan
    
    if len(factor_values) != time_span_hours:
        return np.nan
    
    # Calculate weights
    weights = calculate_wma_weights(time_span_hours, forgetting_factor)
    
    # Calculate weighted average
    factor_values = np.array(factor_values)
    wma_value = np.sum(weights * factor_values)
    
    return wma_value

# Test with some sample data
# Create sample factor data
timestamps = pd.date_range('2024-01-01 10:00:00', '2024-01-01 14:00:00', freq='H')
symbols = ['BTCUSDT', 'ETHUSDT']

# Create sample data
sample_data = []
for ts in timestamps:
    for symbol in symbols:
        # Generate some sample factor values
        value = np.random.uniform(-0.02, 0.02) + 0.01 * np.sin(ts.hour / 24 * 2 * np.pi)
        sample_data.append({'open_time': ts, 'symbol': symbol, 'factor_value': value})

sample_df = pd.DataFrame(sample_data)
sample_df = sample_df.set_index(['open_time', 'symbol'])
print("Sample factor data:")
print(sample_df.head(10))

# Test WMA calculation
current_ts = pd.Timestamp('2024-01-01 12:00:00')
symbol = 'BTCUSDT'
time_span = 3  # 2 hours lookback + current
forgetting_factor = 0.8

wma_result = calculate_wma_for_symbol_timestamp(
    sample_df, symbol, current_ts, time_span, forgetting_factor
)

print(f"\n\nWMA calculation test:")
print(f"Current timestamp: {current_ts}")
print(f"Symbol: {symbol}")
print(f"Time span: {time_span} hours")
print(f"Forgetting factor: {forgetting_factor}")
print(f"WMA result: {wma_result}")

# Show the individual values used
print(f"\nIndividual factor values used:")
for i in range(time_span):
    ts = current_ts - pd.Timedelta(hours=time_span-1-i)
    if ts in sample_df.xs(symbol, level='symbol').index:
        value = sample_df.xs(symbol, level='symbol').loc[ts, 'factor_value']
        weight = calculate_wma_weights(time_span, forgetting_factor)[i]
        print(f"  {ts}: {value:.6f} (weight: {weight:.6f})")

Sample factor data:
                             factor_value
open_time           symbol               
2024-01-01 10:00:00 BTCUSDT     -0.010026
                    ETHUSDT     -0.008388
2024-01-01 11:00:00 BTCUSDT      0.015160
                    ETHUSDT     -0.001880
2024-01-01 12:00:00 BTCUSDT     -0.019356
                    ETHUSDT     -0.019921
2024-01-01 13:00:00 BTCUSDT      0.013629
                    ETHUSDT     -0.012937
2024-01-01 14:00:00 BTCUSDT     -0.021543
                    ETHUSDT      0.004469


WMA calculation test:
Current timestamp: 2024-01-01 12:00:00
Symbol: BTCUSDT
Time span: 3 hours
Forgetting factor: 0.8
WMA result: -0.005591984302675515

Individual factor values used:
  2024-01-01 10:00:00: -0.010026 (weight: 0.262295)
  2024-01-01 11:00:00: 0.015160 (weight: 0.327869)
  2024-01-01 12:00:00: -0.019356 (weight: 0.409836)


/tmp/ipykernel_4062294/2254318018.py:59: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  timestamps = pd.date_range('2024-01-01 10:00:00', '2024-01-01 14:00:00', freq='H')


In [25]:
import os
import pickle
import warnings
from typing import Optional, List, Tuple

def get_month_range(start_date: str, end_date: str) -> List[Tuple[int, int]]:
    """
    Generate a list of (year, month) tuples for the date range.
    
    Args:
        start_date: Start date in format 'YYYY-MM' or 'YYYY-MM-DD'
        end_date: End date in format 'YYYY-MM' or 'YYYY-MM-DD'
    
    Returns:
        List of (year, month) tuples
    """
    # Parse dates
    start = pd.to_datetime(start_date)
    end = pd.to_datetime(end_date)
    
    # Generate month range
    months = []
    current = start.replace(day=1)
    
    while current <= end:
        months.append((current.year, current.month))
        # Move to next month
        if current.month == 12:
            current = current.replace(year=current.year + 1, month=1)
        else:
            current = current.replace(month=current.month + 1)
    
    return months

def load_factor_data_monthly(
    factor_name: str, 
    year: int, 
    month: int,
    factor_dir: str = "./factor_data"
) -> Optional[pd.DataFrame]:
    """
    Load factor data for a specific month.
    
    Args:
        factor_name: Name of the factor directory
        year: Year (e.g., 2024)
        month: Month (1-12)
        factor_dir: Base directory containing factor data
    
    Returns:
        pd.DataFrame: Factor data with MultiIndex (open_time, symbol)
        Returns None if file not found
    """
    try:
        filename = f"factor_{year:04d}-{month:02d}.pkl"
        filepath = os.path.join(factor_dir, factor_name, filename)
        
        if not os.path.exists(filepath):
            warnings.warn(f"Factor file not found: {filepath}")
            return None
        
        # Load pickle file
        with open(filepath, 'rb') as f:
            data = pickle.load(f)
        
        # Ensure proper index format
        if not isinstance(data.index, pd.MultiIndex):
            raise ValueError(f"Expected MultiIndex, got {type(data.index)}")
        
        # Verify index names
        expected_names = ['open_time', 'symbol']
        if data.index.names != expected_names:
            warnings.warn(f"Index names mismatch: expected {expected_names}, got {data.index.names}")
        
        return data
        
    except Exception as e:
        warnings.warn(f"Error loading factor data for {year}-{month:02d}: {e}")
        return None

def save_factor_data_monthly(
    data: pd.DataFrame,
    factor_name: str,
    year: int,
    month: int,
    factor_dir: str = "./factor_data"
) -> bool:
    """
    Save factor data for a specific month.
    
    Args:
        data: Factor data to save
        factor_name: Name of the factor directory
        year: Year
        month: Month
        factor_dir: Base directory to save factor data
    
    Returns:
        bool: True if successful, False otherwise
    """
    try:
        # Create output directory
        output_dir = os.path.join(factor_dir, factor_name)
        os.makedirs(output_dir, exist_ok=True)
        
        # Create filename
        filename = f"factor_{year:04d}-{month:02d}.pkl"
        filepath = os.path.join(output_dir, filename)
        
        # Save to pickle file
        with open(filepath, 'wb') as f:
            pickle.dump(data, f, protocol=pickle.HIGHEST_PROTOCOL)
        
        print(f"[SAVE] Saved to: {filepath}")
        return True
        
    except Exception as e:
        warnings.warn(f"Error saving factor data for {year}-{month:02d}: {e}")
        return False

def load_factor_data_range(
    factor_name: str,
    start_date: str,
    end_date: str,
    factor_dir: str = "./factor_data"
) -> Optional[pd.DataFrame]:
    """
    Load factor data for a date range, combining multiple months.
    
    Args:
        factor_name: Name of the factor directory
        start_date: Start date in format 'YYYY-MM' or 'YYYY-MM-DD'
        end_date: End date in format 'YYYY-MM' or 'YYYY-MM-DD'
        factor_dir: Base directory containing factor data
    
    Returns:
        pd.DataFrame: Combined factor data with MultiIndex (open_time, symbol)
    """
    months = get_month_range(start_date, end_date)
    all_data = []
    
    print(f"Loading factor data: {factor_name}")
    print(f"Date range: {start_date} to {end_date}")
    print(f"Months to load: {len(months)}")
    
    for year, month in months:
        print(f"  Loading {year}-{month:02d}...")
        monthly_data = load_factor_data_monthly(factor_name, year, month, factor_dir)
        
        if monthly_data is not None:
            all_data.append(monthly_data)
            print(f"    Loaded {len(monthly_data):,} rows")
        else:
            print(f"    Failed to load")
    
    if len(all_data) == 0:
        warnings.warn(f"No factor data found for {factor_name} in range {start_date} to {end_date}")
        return None
    
    # Combine all months
    combined = pd.concat(all_data, axis=0)
    combined = combined.sort_index()
    
    print(f"Combined data: {len(combined):,} rows, {len(combined.index.get_level_values('symbol').unique())} symbols")
    return combined

# Test the loading function
print("Testing factor data loading...")
test_factor_data = load_factor_data_range(
    factor_name="cnn_04_09_16h_v2",
    start_date="2024-01",
    end_date="2024-01",
    factor_dir="/home/craz/crypto/crypto-trading/strategy/backtest/factor_data"
)

if test_factor_data is not None:
    print(f"\nSuccessfully loaded test data:")
    print(f"Shape: {test_factor_data.shape}")
    print(f"Date range: {test_factor_data.index.get_level_values('open_time').min()} to {test_factor_data.index.get_level_values('open_time').max()}")
    print(f"Symbols: {len(test_factor_data.index.get_level_values('symbol').unique())}")
else:
    print("Failed to load test data")

Testing factor data loading...
Loading factor data: cnn_04_09_16h_v2
Date range: 2024-01 to 2024-01
Months to load: 1
  Loading 2024-01...
    Loaded 174,937 rows
Combined data: 174,937 rows, 240 symbols

Successfully loaded test data:
Shape: (174937, 1)
Date range: 2024-01-01 00:00:00 to 2024-01-31 23:00:00
Symbols: 240
